> Legacy/optional workflow. For author SLM-SQL + CSC baseline and CSC LoRA research, use **09 → 10 → 11** and `docs/CSC_LORA.md`. This notebook is not the canonical comparison against the author system.

# 03 — LoRA SFT Training

Train LoRA adapters on filtered SynSQL data using Unsloth + TRL.
Grid search over rank, target modules, learning rate, RSLoRA.

In [ ]:
import time
import json
import os
import subprocess
import sys
from pathlib import Path


def bootstrap_repo():
    configured = os.environ.get('SLM_PRJ_PATH')
    candidates = [Path(configured).expanduser()] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents,
                       Path('/kaggle/working/SLM-PRJ'),
                       Path('/kaggle/working/SLMSQL_project')])
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'benchmarks').is_dir():
            return candidate
    target = Path('/kaggle/working/SLM-PRJ')
    url = os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git')
    subprocess.run(['git', 'clone', url, str(target)], check=True)
    return target

REPO_DIR = bootstrap_repo()
sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_utils import prepare_repo_import, resolve_data_dir
REPO_DIR = prepare_repo_import(REPO_DIR)
DATA_DIR = resolve_data_dir(REPO_DIR)

RESULTS_DIR = Path(os.environ.get('LORA_OUTPUT_DIR', str(REPO_DIR / 'results' / 'lora')))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
from scripts.kaggle_utils import load_kaggle_secrets, publish_checkpoint
from training.lora_config import generate_experiments
from training.sft_trainer import train
from training.merge_adapter import merge_and_save
print(f'Repo: {REPO_DIR}')
print(f'Data: {DATA_DIR}')

## Grid Search on 5K Subset

In [ ]:
MODEL_KEY = os.environ.get('LORA_MODEL_KEY', 'small')
all_experiments = generate_experiments(REPO_DIR / 'configs/lora_grid.yaml', MODEL_KEY)
experiments = all_experiments
max_experiments = int(os.environ.get('MAX_EXPERIMENTS', '1'))
if os.environ.get('RUN_FULL_GRID', '0') != '1':
    experiments = experiments[:max_experiments]
print(f'{len(experiments)} experiment(s) selected (set RUN_FULL_GRID=1 for the full grid)')
for e in experiments:
    print(f'  {e.name}: rank={e.rank}, modules={e.target_modules}, lr={e.learning_rate}, rslora={e.rslora}')


In [ ]:
TRAIN_DATA = Path(os.environ.get('TRAIN_DATA', str(DATA_DIR / 'sft_subsets/sft_10000.jsonl')))
if not TRAIN_DATA.is_file():
    raise FileNotFoundError(f'Run 01 or set TRAIN_DATA: {TRAIN_DATA}')
COT_MODE = os.environ.get('COT_MODE', 'full_cot')
PILOT_SAMPLES = 5000
grid_results = []
for exp in experiments:
    output_dir = RESULTS_DIR / exp.name / COT_MODE / 'pilot'
    command = [sys.executable, '-m', 'training.sft_trainer',
               '--model-key', MODEL_KEY, '--experiment-index', str(all_experiments.index(exp)),
               '--train-data', str(TRAIN_DATA), '--output-dir', str(output_dir),
               '--max-samples', str(PILOT_SAMPLES), '--cot-mode', COT_MODE]
    if os.environ.get('RESUME_FROM'):
        if len(experiments) != 1:
            raise ValueError('Explicit RESUME_FROM is valid for one configuration only.')
        command += ['--resume-from', os.environ['RESUME_FROM']]
    try:
        # Each model is released with its process, including optimizer and CUDA state.
        subprocess.run(command, cwd=REPO_DIR, check=True)
        result = json.loads((output_dir / 'train_stats.json').read_text())
        grid_results.append({'name': exp.name, **result})
    except subprocess.CalledProcessError as exc:
        grid_results.append({'name': exp.name, 'error': str(exc)})
    (RESULTS_DIR / 'grid_results.json').write_text(json.dumps(grid_results, indent=2))


## Quick Eval on 300 Dev Questions

In [ ]:
BIRD_DIR = DATA_DIR / 'bird'
SPLIT_MANIFEST = REPO_DIR / 'results/legacy_protocol/data_split.json'
successful = [r for r in grid_results if 'error' not in r and r.get('adapter_path')]
print({'successful_pilots': len(successful), 'selection': 'calibration EX, then measured training time'})


In [ ]:
evaluated = []
for result in successful:
    adapter = Path(result['adapter_path'])
    merged = adapter.parent / 'merged'
    try:
        subprocess.run([sys.executable, '-m', 'training.merge_adapter',
                        '--base-model', result['config']['model_id'], '--adapter-path', str(adapter),
                        '--output-path', str(merged)], cwd=REPO_DIR, check=True)
        eval_file = adapter.parent / 'calibration_eval.json'
        subprocess.run([sys.executable, '-m', 'training.evaluate_checkpoint',
                        '--model', str(merged), '--data-dir', str(BIRD_DIR),
                        '--split-manifest', str(SPLIT_MANIFEST), '--output', str(eval_file),
                        '--limit', '300'], cwd=REPO_DIR, check=True)
        metrics = json.loads(eval_file.read_text())
        evaluated.append({**result, 'calibration_ex': metrics['accuracy'], 'merged_path': str(merged)})
    except subprocess.CalledProcessError as exc:
        print('Pilot evaluation failed:', result['name'], exc)
if evaluated:
    top_result = max(evaluated, key=lambda r: (r['calibration_ex'], -r['wall_time_s']))
    (RESULTS_DIR / 'best_checkpoint.json').write_text(json.dumps({
        'experiment': top_result['name'], 'adapter_path': top_result['adapter_path'],
        'merged_path': top_result['merged_path'], 'model_id': top_result['config']['model_id'],
        'calibration_ex': top_result['calibration_ex'], 'stage': 'pilot', 'cot_mode': COT_MODE,
    }, indent=2))
    print('Selected pilot:', top_result['name'], 'calibration EX:', top_result['calibration_ex'])
else:
    top_result = None
    print('No successfully evaluated pilot; no best checkpoint was selected.')


## Full Training (30-50K samples)

Run with the best config from the grid search above.

In [ ]:
# A separate full-training directory prevents mixing a 5K pilot with 30K/50K runs.
if os.environ.get('RUN_FULL_TRAIN', '0') == '1':
    if top_result is None:
        raise ValueError('Select a pilot configuration before full training.')
    full_data = Path(os.environ.get('FULL_TRAIN_DATA', str(DATA_DIR / 'sft_subsets/sft_30000.jsonl')))
    output = RESULTS_DIR / top_result['name'] / COT_MODE / full_data.stem
    exp = next(e for e in all_experiments if e.name == top_result['name'])
    subprocess.run([sys.executable, '-m', 'training.sft_trainer', '--model-key', MODEL_KEY,
                    '--experiment-index', str(all_experiments.index(exp)), '--train-data', str(full_data),
                    '--output-dir', str(output), '--cot-mode', COT_MODE], cwd=REPO_DIR, check=True)
    full_stats = json.loads((output / 'train_stats.json').read_text())
    merged = output / 'merged'
    subprocess.run([sys.executable, '-m', 'training.merge_adapter', '--base-model', exp.model_id,
                    '--adapter-path', full_stats['adapter_path'], '--output-path', str(merged)],
                   cwd=REPO_DIR, check=True)
    (RESULTS_DIR / 'best_checkpoint.json').write_text(json.dumps({
        'experiment': exp.name, 'model_id': exp.model_id, 'adapter_path': full_stats['adapter_path'],
        'merged_path': str(merged), 'stage': 'full', 'train_data': str(full_data), 'cot_mode': COT_MODE,
    }, indent=2))
else:
    print('Full training is off. Set RUN_FULL_TRAIN=1 and FULL_TRAIN_DATA after pilot selection.')


In [ ]:
# Optional: publish one adapter/merged checkpoint to a private Hugging Face model repo.
# In Kaggle, add HF_TOKEN and HF_REPO_ID as Secrets; values are never printed.
load_kaggle_secrets()
checkpoint_raw = os.environ.get('CHECKPOINT_DIR', '')
if not checkpoint_raw and 'merge_path' in globals() and Path(merge_path).is_dir():
    checkpoint_raw = str(merge_path)
if checkpoint_raw:
    print(publish_checkpoint(checkpoint_raw))
else:
    print('No checkpoint selected. Set CHECKPOINT_DIR before publishing.')